1. Objetivo

00 — Environment Validation

Objetivo: validar infraestrutura local do Contract Lakehouse.

Valida:
- SparkSession
- conexão com Spark Master
- bucket MinIO
- escrita Parquet via S3A
- leitura Parquet via S3A

2. Setup Spark

In [1]:
from src.core.session import SparkFactory
import boto3

spark = SparkFactory.create()

print(f"Spark version: {spark.version}")

26/06/25 03:00:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 03:00:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.1


3. Validar MinIO

In [2]:
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="admin12345",
    region_name="us-east-1",
)

bucket_name = "contracts"

existing_buckets = [
    bucket["Name"]
    for bucket in s3_client.list_buckets()["Buckets"]
]

if bucket_name not in existing_buckets:
    s3_client.create_bucket(Bucket=bucket_name)

print(f"Buckets disponíveis: {existing_buckets}")

Buckets disponíveis: ['contracts']


4. Validar escrita Parquet

In [3]:
data = [
    ("5900055119", "Contrato A", "1000000.0"),
    ("5900086165", "Contrato B", "2500000.0"),
]

df = spark.createDataFrame(
    data,
    ["contract_number", "contract_name", "contract_value"]
)

output_path = "s3a://contracts/test/environment_validation/"

df.write.mode("overwrite").parquet(output_path)

print(f"Arquivo gravado em: {output_path}")

26/06/25 03:00:44 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Arquivo gravado em: s3a://contracts/test/environment_validation/


5. Validar leitura Parquet

In [4]:
validation_df = spark.read.parquet(output_path)

validation_df.show()
validation_df.printSchema()

+---------------+-------------+--------------+
|contract_number|contract_name|contract_value|
+---------------+-------------+--------------+
|     5900055119|   Contrato A|     1000000.0|
|     5900086165|   Contrato B|     2500000.0|
+---------------+-------------+--------------+

root
 |-- contract_number: string (nullable = true)
 |-- contract_name: string (nullable = true)
 |-- contract_value: string (nullable = true)



6. Sumário

In [5]:
environment_validation = {
    "spark_version": spark.version,
    "bucket": bucket_name,
    "parquet_path": output_path,
    "rows_read": validation_df.count(),
    "status": "SUCCESS"
}

environment_validation

{'spark_version': '3.5.1',
 'bucket': 'contracts',
 'parquet_path': 's3a://contracts/test/environment_validation/',
 'rows_read': 2,
 'status': 'SUCCESS'}

7. Stop Spark - Isso libera o worker para o próximo notebook.

In [6]:
spark.stop()